# Hull Tactical Market Prediction - Ensemble Models
Este notebook implementa varios métodos de ensemble para predecir retornos del mercado

## Imports

In [ ]:
import os
from pathlib import Path
import datetime

from tqdm import tqdm
from dataclasses import dataclass, asdict

import polars as pl 
import numpy as np

# Modelos base
from sklearn.neighbors import KNeighborsRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.svm import SVR
from sklearn.linear_model import Ridge, Lasso

# Métodos de ensemble
from sklearn.ensemble import (
    VotingRegressor,
    StackingRegressor,
    BaggingRegressor,
    AdaBoostRegressor,
    RandomForestRegressor
)

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

import kaggle_evaluation.default_inference_server

## Project Directory Structure

In [ ]:
import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

## Configurations

In [ ]:
# ============ PATHS ============
DATA_PATH: Path = Path('/kaggle/input/hull-tactical-market-prediction/')

# ============ RETURNS TO SIGNAL CONFIGS ============
MIN_SIGNAL: float = 0.0                        
MAX_SIGNAL: float = 2.0                         
SIGNAL_MULTIPLIER: float = 400.0                

# ============ ENSEMBLE MODEL CONFIGS ============
N_ESTIMATORS: int = 50
MAX_SAMPLES: float = 0.6
RANDOM_STATE: int = 42

## Data Classes

In [ ]:
@dataclass
class DatasetOutput:
    X_train : pl.DataFrame 
    X_test: pl.DataFrame
    y_train: pl.Series
    y_test: pl.Series
    scaler: StandardScaler

@dataclass(frozen=True)
class RetToSignalParameters:
    signal_multiplier: float 
    min_signal : float = MIN_SIGNAL
    max_signal : float = MAX_SIGNAL

In [ ]:
ret_signal_params = RetToSignalParameters(
    signal_multiplier=SIGNAL_MULTIPLIER
)

## Data Loading and Preprocessing Functions

In [ ]:
def load_trainset() -> pl.DataFrame:
    return (
        pl.read_csv(DATA_PATH / "train.csv")
        .rename({'market_forward_excess_returns':'target'})
        .with_columns(
            pl.exclude('date_id').cast(pl.Float64, strict=False)
        )
        .head(-10)
    )

def load_testset() -> pl.DataFrame:
    return (
        pl.read_csv(DATA_PATH / "test.csv")
        .rename({'lagged_forward_returns':'target'})
        .with_columns(
            pl.exclude('date_id').cast(pl.Float64, strict=False)
        )
    )

def create_example_dataset(df: pl.DataFrame) -> pl.DataFrame:
    vars_to_keep = [
        "S2", "E2", "E3", "P9", "S1", "S5", "I2", "P8",
        "P10", "P12", "P13", "U1", "U2"
    ]

    return (
        df.with_columns(
            (pl.col("I2") - pl.col("I1")).alias("U1"),
            (pl.col("M11") / ((pl.col("I2") + pl.col("I9") + pl.col("I7")) / 3)).alias("U2")
        )
        .select(["date_id", "target"] + vars_to_keep)
        .with_columns([
            pl.col(col).fill_null(pl.col(col).ewm_mean(com=0.5))
            for col in vars_to_keep
        ])
        .drop_nulls()
    )
    
def join_train_test_dataframes(train: pl.DataFrame, test: pl.DataFrame) -> pl.DataFrame:
    common_columns = [col for col in train.columns if col in test.columns]
    return pl.concat([train.select(common_columns), test.select(common_columns)], how="vertical")

def split_dataset(train: pl.DataFrame, test: pl.DataFrame, features: list) -> DatasetOutput: 
    X_train = train.drop(['date_id','target']) 
    y_train = train.get_column('target')
    X_test = test.drop(['date_id','target']) 
    y_test = test.get_column('target')
    
    scaler = StandardScaler() 
    
    X_train_scaled_np = scaler.fit_transform(X_train)
    X_train = pl.from_numpy(X_train_scaled_np, schema=features)
    
    X_test_scaled_np = scaler.transform(X_test)
    X_test = pl.from_numpy(X_test_scaled_np, schema=features)
    
    return DatasetOutput(
        X_train=X_train,
        y_train=y_train, 
        X_test=X_test, 
        y_test=y_test,
        scaler=scaler
    )

def convert_ret_to_signal(
    ret_arr: np.ndarray,
    params: RetToSignalParameters
) -> np.ndarray:
    return np.clip(
        ret_arr * params.signal_multiplier + 1, params.min_signal, params.max_signal
    )

## Load and Prepare Data

In [ ]:
train = load_trainset()
test = load_testset() 
print(train.tail(3)) 
print(test.head(3))

In [ ]:
df = join_train_test_dataframes(train, test)
df = create_example_dataset(df=df) 
train = df.filter(pl.col('date_id').is_in(train.get_column('date_id')))
test = df.filter(pl.col('date_id').is_in(test.get_column('date_id')))

FEATURES = [col for col in test.columns if col not in ['date_id', 'target']]

dataset = split_dataset(train=train, test=test, features=FEATURES) 

X_train = dataset.X_train
X_test = dataset.X_test
y_train = dataset.y_train
y_test = dataset.y_test
scaler = dataset.scaler

print(f"Training set shape: {X_train.shape}")
print(f"Test set shape: {X_test.shape}")

## Model Evaluation Function

In [ ]:
def evaluate_model(model, X_test, y_test, model_name):
    """Evalúa el rendimiento de un modelo"""
    y_pred = model.predict(X_test)
    
    mse = mean_squared_error(y_test, y_pred)
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)
    
    print(f"\n{'='*50}")
    print(f"Resultados para {model_name}")
    print(f"{'='*50}")
    print(f"RMSE: {rmse:.6f}")
    print(f"MAE:  {mae:.6f}")
    print(f"R²:   {r2:.6f}")
    
    return {'model': model_name, 'rmse': rmse, 'mae': mae, 'r2': r2}

## 1. Voting Ensemble (Votación)

In [ ]:
# Crear modelos base
knn_model = KNeighborsRegressor(n_neighbors=5)
tree_model = DecisionTreeRegressor(max_depth=10, random_state=RANDOM_STATE)
svr_model = SVR(kernel='rbf')

# Crear ensamble de votación
voting_model = VotingRegressor(
    estimators=[
        ('KNN', knn_model),
        ('Tree', tree_model),
        ('SVR', svr_model)
    ]
)

# Entrenar
print("Entrenando Voting Ensemble...")
voting_model.fit(X_train, y_train)

# Evaluar
voting_results = evaluate_model(voting_model, X_test, y_test, "Voting Ensemble")

## 2. Stacking Ensemble (Apilamiento)

In [ ]:
# Modelos de primer nivel
knn_stack = KNeighborsRegressor(n_neighbors=5)
tree_stack = DecisionTreeRegressor(max_depth=10, random_state=RANDOM_STATE)
svr_stack = SVR(kernel='rbf')

# Modelo de segundo nivel (meta-modelo)
meta_model = Ridge(alpha=1.0)

# Crear ensamble de apilamiento
stacking_model = StackingRegressor(
    estimators=[
        ('KNN', knn_stack),
        ('Tree', tree_stack),
        ('SVR', svr_stack)
    ],
    final_estimator=meta_model
)

# Entrenar
print("Entrenando Stacking Ensemble...")
stacking_model.fit(X_train, y_train)

# Evaluar
stacking_results = evaluate_model(stacking_model, X_test, y_test, "Stacking Ensemble")

## 3. Bagging Ensemble

In [ ]:
# Modelo base para bagging
base_tree = DecisionTreeRegressor(max_depth=10, random_state=RANDOM_STATE)

# Crear ensamble de bagging
bagging_model = BaggingRegressor(
    estimator=base_tree,
    n_estimators=N_ESTIMATORS,
    max_samples=MAX_SAMPLES,
    random_state=RANDOM_STATE,
    n_jobs=-1
)

# Entrenar
print("Entrenando Bagging Ensemble...")
bagging_model.fit(X_train, y_train)

# Evaluar
bagging_results = evaluate_model(bagging_model, X_test, y_test, "Bagging Ensemble")

## 4. Boosting Ensemble (AdaBoost)

In [ ]:
# Modelo base para boosting
base_boost = DecisionTreeRegressor(max_depth=5, random_state=RANDOM_STATE)

# Crear ensamble de boosting
boosting_model = AdaBoostRegressor(
    estimator=base_boost,
    n_estimators=N_ESTIMATORS,
    random_state=RANDOM_STATE,
    learning_rate=0.1
)

# Entrenar
print("Entrenando Boosting Ensemble...")
boosting_model.fit(X_train, y_train)

# Evaluar
boosting_results = evaluate_model(boosting_model, X_test, y_test, "Boosting Ensemble (AdaBoost)")

## 5. Random Forest

In [ ]:
# Crear Random Forest
rf_model = RandomForestRegressor(
    n_estimators=N_ESTIMATORS,
    max_depth=10,
    max_features='sqrt',
    random_state=RANDOM_STATE,
    n_jobs=-1
)

# Entrenar
print("Entrenando Random Forest...")
rf_model.fit(X_train, y_train)

# Evaluar
rf_results = evaluate_model(rf_model, X_test, y_test, "Random Forest")

## Comparación de Modelos

In [ ]:
# Crear tabla comparativa
results_df = pl.DataFrame([
    voting_results,
    stacking_results,
    bagging_results,
    boosting_results,
    rf_results
])

print("\n" + "="*70)
print("COMPARACIÓN DE TODOS LOS MODELOS")
print("="*70)
print(results_df.sort('rmse'))

# Seleccionar el mejor modelo
best_model_name = results_df.sort('rmse')[0, 'model']
print(f"\n✓ Mejor modelo: {best_model_name}")

## Selección del Modelo Final para Producción

Basándonos en los resultados, seleccionaremos el mejor modelo.

In [ ]:
# Entrenar el mejor modelo con todos los datos disponibles
# Para este ejemplo, usaremos Random Forest que generalmente funciona bien

final_model = RandomForestRegressor(
    n_estimators=100,
    max_depth=12,
    max_features='sqrt',
    random_state=RANDOM_STATE,
    n_jobs=-1
)

print("Entrenando modelo final con todos los datos...")
final_model.fit(X_train, y_train)

print("✓ Modelo final entrenado y listo para producción")

## Función de Predicción para Kaggle

In [ ]:
def predict(test: pl.DataFrame) -> float:
    """Función de predicción para el servidor de inferencia de Kaggle"""
    test = test.rename({'lagged_forward_returns':'target'})
    df = create_example_dataset(test)
    X_test = df.select(FEATURES)
    X_test_scaled_np = scaler.transform(X_test)
    X_test = pl.from_numpy(X_test_scaled_np, schema=FEATURES)
    raw_pred = final_model.predict(X_test)[0]
    return convert_ret_to_signal(raw_pred, ret_signal_params)

## Configuración del Servidor de Inferencia

In [ ]:
inference_server = kaggle_evaluation.default_inference_server.DefaultInferenceServer(predict)

if os.getenv('KAGGLE_IS_COMPETITION_RERUN'):
    inference_server.serve()
else:
    inference_server.run_local_gateway(('/kaggle/input/hull-tactical-market-prediction/',))